# HotpotQA 多义词语义传播数据集构建

从 HotpotQA 构建一个带 gold 语义标签的多义词数据集，用来评测 LiteSemRAG 的无训练语义传播算法（Anchor-C/D/E/F 等）。

构建流程：

1. **多义词来源**：加载 `hotpotqa_latest_framework_index.ipynb` 产出的 LiteSemRAG 索引（`litesemrag_hotpotqa_500.pkl`），取 `len(token_node.sem_node_list) >= 2` 的 token 的 surface form 作为「多义词候选」。这里**只取词表**，不使用该索引内部的义项切分结果。
2. **选词**：按这些词在 scan store 中的 occurrence 频次排序，取 Top-K。
3. **取样本**：对每个词从 scan store（`hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl`）捞最多 `MAX_OCCURRENCES_PER_WORD` 条 occurrence，用 DeBERTa 取 span embedding（与 LiteSemRAG / `hotpotqa_fft_span_compare.ipynb` 同款）。
4. **gold 标注（参考 LiteSemRAG）**：先采 anchor（FFT + random），用 anchor 上下文引导 LLM **合并/筛选** Wikidata 候选义项得到每词一个 merged candidate bank；再用 LLM 对**全部** occurrence 逐条在该 bank 上做义项分配，作为 gold（true label）。
5. **落盘**：每条样本保存 span embedding + gold 标签 + 上下文/span/anchor 信息，供后续 anchor 传播对比实验离线复算。

> 多数 helper 函数与 `hotpotqa_anchor_propagation_compare.ipynb` 一致，以保证 embedding / 候选合并 / gold 标注口径完全对齐。


In [1]:
from pathlib import Path
import os
import sys

# 定位项目根目录，并切换工作目录、加入 Python 搜索路径
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")

Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import pickle
import re
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModel, AutoTokenizer

import RAG_graph
from text_processing import get_token_indices_for_phrase, normalize_text
from utils import (
    build_wikidata_candidate_bank as shared_build_wikidata_candidate_bank,
    farthest_first_traversal,
    load_wikidata_definition_candidates as shared_load_wikidata_definition_candidates,
)
from wikidata_definition_filter import (
    MERGE_WITH_SAMPLES_SYSTEM_PROMPT,
    MERGE_WITH_SAMPLES_USER_PROMPT_TEMPLATE,
    WIKIDATA_CANDIDATE_MERGE_WITH_SAMPLES_VERSION,
    CandidateSense,
    WikidataDefinitionFilter,
    _build_definitions,
    _extract_json_object,
)
from llm_semantic_labeler import (
    DEFAULT_CACHE_PATH as DEFAULT_LLM_CACHE_PATH,
    DEFAULT_CONTEXT_WORD_WINDOW as DEFAULT_LLM_CONTEXT_WORD_WINDOW,
    DEFAULT_MAX_TOKENS as DEFAULT_LLM_MAX_TOKENS,
    choose_wikidata_candidate_with_llm,
    choose_wikidata_candidates_with_llm_batch,
)
from local_llm import LocalLLMClient, LocalLLMConfig


In [3]:
# =============================================================================
# 全局参数（修改本单元格后自上而下重新运行）
# =============================================================================

# --- 设备与模型 ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_TEXT_ENCODER_NAME_OR_PATH = "/home/xiaoyue/ProtoGraphRAG/deberta-v3-large"  # span 嵌入用 DeBERTa

# --- 多义词来源：LiteSemRAG 索引（latest_framework_index 产物）---
INDEX_PKL_PATH = REPO_ROOT / "cache" / "hotpotqa_latest_framework_index" / "litesemrag_hotpotqa_500.pkl"
MULTI_SEM_MIN_SEM_COUNT = 2  # token 至少有这么多 sem node 才算多义词候选

# --- scan store：occurrence 来源 ---
HOTPOT_SCAN_STORE_PATH = REPO_ROOT / "data/hotpot_QA_qwen_scan_cache" / "hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl"
QUERY_KIND = None  # None 表示 phrase+token 全部保留

# --- 选词与样本规模 ---
TOP_K_WORDS = 50  # 按 occurrence 频次取前 K 个多义词
MAX_OCCURRENCES_PER_WORD = 100  # 每个词最多取多少条 occurrence 进完整数据集
MIN_OCCURRENCES_FOR_DATASET = 20  # scan store 中 occurrence 少于此数的词不入选（样本太少没法评测传播）

# --- 单词级试跑 ---
RUN_SINGLE_WORD_TEST = False  # True 时只为 TEST_WORD 先构建一个小样本测试集并展示摘要
TEST_WORD = "director"  # None 表示使用 selected_words[0]；也可以手动填入如 "film"
TEST_MAX_OCCURRENCES_PER_WORD = 50  # 单词试跑最多取多少条 occurrence，控制 LLM 标注成本
TEST_MIN_OCCURRENCES_FOR_DATASET = 5  # 单词试跑的最低样本数门槛
RUN_FULL_DATASET_BUILD = True  # True 才执行后面的完整 selected_words 数据集构建与落盘

# --- 嵌入与上下文 ---
DEFAULT_BATCH_SIZE = 16
DEFAULT_MAX_LENGTH = 512
DEFAULT_CONTEXT_MODE = "sentence_neighbors"  # sentence / sentence_neighbors / full_text
MARK_TARGET = False

# --- anchor 采样（gold 候选合并的样本来源；与 anchor 传播评测口径一致）---
ANCHOR_FRACTION = 0.15  # anchor 占全部 occurrence 的比例
ANCHOR_FFT_RATIO = 0.70  # anchor 中由 FFT 选出的比例（其余 random）
ANCHOR_MIN_COUNT = 2  # anchor 最少个数
ANCHOR_RANDOM_STATE = 42  # anchor 随机采样种子

# --- Wikidata 候选 + LLM 合并（wikidata_llm_candidate_merge_experiment 同款）---
WIKIDATA_CANDIDATE_LIMIT = 8  # Wikidata 原始候选上限
USE_DETAILED_DESCRIPTION = False
USE_LLM_WIKIDATA = True  # True：anchor 引导的 LLM merge_with_samples；False：仅规则过滤 Wikidata
LLM_WIKIDATA_USE_API = True  # True=DeepSeek API；False=本地 OpenAI 兼容服务
LLM_WIKIDATA_PROVIDER = "deepseek"
LLM_WIKIDATA_MODEL = None  # DeepSeek provider 默认使用 local_llm.DEFAULT_DEEPSEEK_MODEL
LLM_WIKIDATA_API_KEY_FILE = "API_KEY"  # 读取 DEEPSEEK_API_KEY / deepseek_api
LLM_WIKIDATA_CACHE_PATH = REPO_ROOT / "cache" / "wikidata_definition_filter_cache_deepseek.sqlite3"

# 仅 USE_LLM_WIKIDATA=False 的规则过滤路径用到
RULE_EXACT_MATCH_TEXT = True
RULE_FILTER_NAME = False
RULE_REQUIRE_DETAILED_DESCRIPTION = True

# --- gold：LLM 全量逐条标注 ---
LLM_PROVIDER = "deepseek"
LLM_MODEL = None  # DeepSeek provider 默认使用 local_llm.DEFAULT_DEEPSEEK_MODEL
LLM_API_KEY_FILE = "API_KEY"  # 读取 DEEPSEEK_API_KEY / deepseek_api
LLM_CACHE_PATH = REPO_ROOT / "cache" / "llm_semantic_label_cache_deepseek.sqlite3"
LLM_CONTEXT_WORD_WINDOW = DEFAULT_LLM_CONTEXT_WORD_WINDOW
LLM_MAX_TOKENS = 1024  # batch 模式需要一次返回多条 judgment，避免 JSON 被截断
LLM_LABEL_PROMPT_MODE = "batch"  # batch 使用 llm_semantic_labeler.BATCH_* 模板；single 为逐样本模板
LLM_LABEL_BATCH_SIZE = 10  # batch 模式下每个 prompt 放入多少条 occurrence

# --- 输出 ---
DATASET_OUTPUT_DIR = REPO_ROOT / "data" / "polysemy_sem_eval"
LLM_RUN_TAG = "deepseek_v4_flash"
DATASET_RUN_ID = time.strftime("%Y%m%d_%H%M%S")  # 每次从参数 cell 开始重跑都会生成新版本
DATASET_TAG = f"{LLM_RUN_TAG}_top{TOP_K_WORDS}_max{MAX_OCCURRENCES_PER_WORD}"
DATASET_VERSION_TAG = f"{DATASET_TAG}_{DATASET_RUN_ID}"
DATASET_PKL_PATH = DATASET_OUTPUT_DIR / f"hotpotqa_polysemy_dataset_{DATASET_VERSION_TAG}.pkl"
DATASET_SUMMARY_CSV_PATH = DATASET_OUTPUT_DIR / f"hotpotqa_polysemy_dataset_{DATASET_VERSION_TAG}_summary.csv"
SAVED_DATASET_PKL_PATH = None
SAVED_DATASET_SUMMARY_CSV_PATH = None
DATASET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", None)
print(f"Device: {DEVICE}")
print(f"Index (multi-sem word source): {INDEX_PKL_PATH}")
print(f"Scan store (occurrence source): {HOTPOT_SCAN_STORE_PATH}")
print(f"Top-K words: {TOP_K_WORDS} | max occurrences/word: {MAX_OCCURRENCES_PER_WORD}")
print(f"LLM provider: {LLM_PROVIDER} | Wikidata merge provider: {LLM_WIKIDATA_PROVIDER}")
print(f"Gold labeling prompt mode: {LLM_LABEL_PROMPT_MODE} | batch size: {LLM_LABEL_BATCH_SIZE}")
print(
    f"Single-word test: {RUN_SINGLE_WORD_TEST} | word={TEST_WORD or 'selected_words[0]'} | "
    f"max occurrences={TEST_MAX_OCCURRENCES_PER_WORD} | full build={RUN_FULL_DATASET_BUILD}"
)
print(f"Dataset version: {DATASET_VERSION_TAG}")
print(f"Dataset output: {DATASET_PKL_PATH}")


Device: cuda
Index (multi-sem word source): /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_500.pkl
Scan store (occurrence source): /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
Top-K words: 50 | max occurrences/word: 100
LLM provider: deepseek | Wikidata merge provider: deepseek
Gold labeling prompt mode: batch | batch size: 10
Single-word test: False | word=director | max occurrences=50 | full build=True
Dataset version: deepseek_v4_flash_top50_max100_20260605_142830
Dataset output: /home/xiaoyue/LiteSemRAG/data/polysemy_sem_eval/hotpotqa_polysemy_dataset_deepseek_v4_flash_top50_max100_20260605_142830.pkl


In [4]:
def load_hotpot_scan_store(scan_store_path: Path = HOTPOT_SCAN_STORE_PATH):
    if not scan_store_path.exists():
        raise FileNotFoundError(
            f"HotpotQA scan store not found at {scan_store_path}. Please prepare the scan cache first."
        )
    with scan_store_path.open("rb") as handle:
        store = pickle.load(handle)
    print(f"Loaded scan-only store from {scan_store_path}")
    print(store["stats"])
    return store


embedding_store = load_hotpot_scan_store()

Loaded scan-only store from /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
{'num_documents': 4937, 'num_unique_terms': 60797, 'num_phrase_occurrences': 77703, 'num_token_occurrences': 93649, 'num_total_occurrences': 171352}


In [5]:
def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def _find_left_boundary(text: str, index: int) -> int:
    return max(
        text.rfind(".", 0, index),
        text.rfind("!", 0, index),
        text.rfind("?", 0, index),
    )


def _find_right_boundary(text: str, index: int) -> int:
    right_candidates = [
        text.find(".", index),
        text.find("!", index),
        text.find("?", index),
    ]
    right_candidates = [idx for idx in right_candidates if idx != -1]
    return len(text) if not right_candidates else min(right_candidates) + 1


def _build_context_from_bounds(cleaned_text, span, context_start, context_end):
    start_char, end_char = span
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {"context_text": context_text, "local_span": (local_start, local_end)}


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)
    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_neighbor_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)

    if context_start > 0:
        previous_boundary = _find_left_boundary(cleaned_text, max(0, context_start - 1))
        context_start = 0 if previous_boundary == -1 else previous_boundary + 1

    if context_end < len(cleaned_text):
        context_end = _find_right_boundary(cleaned_text, context_end)

    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_full_context(cleaned_text, span):
    start_char, end_char = span
    context_raw = cleaned_text
    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - left_trim
    local_end = end_char - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {"context_text": context_text, "local_span": (local_start, local_end)}


def extract_prompt_context(cleaned_text, span, prompt_context_mode="sentence"):
    if prompt_context_mode == "sentence":
        return extract_sentence_context(cleaned_text, span)
    if prompt_context_mode == "full_text":
        return extract_full_context(cleaned_text, span)
    if prompt_context_mode == "sentence_neighbors":
        return extract_neighbor_sentence_context(cleaned_text, span)
    raise ValueError(
        f"Unsupported prompt_context_mode={prompt_context_mode!r}. Use 'sentence', 'sentence_neighbors', or 'full_text'."
    )


def build_hotpot_prompt(
    record,
    query_text,
    mark_target=False,
    left_marker="[TGT]",
    right_marker="[/TGT]",
    prompt_context_mode="sentence",
):
    context_info = extract_prompt_context(
        record["cleaned_text"], record["span"], prompt_context_mode=prompt_context_mode
    )
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} {context_text[local_start:local_end]} {right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Context: {prompt_context}\n"
        f"Target word: {query_text.strip()}\n\n"
        f'Question: What does "{query_text.strip()}" mean in this context?'
    )

    return {
        "context_text": context_text,
        "matched_text": context_text[local_start:local_end],
        "local_span": (local_start, local_end),
        "prompt_text": prompt_text,
    }

In [6]:
def load_wikidata_definition_candidates(
    query_text: str,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
    filter_name: bool = True,
    require_detailed_description: bool = False,
    candidate_limit: int = 5,
    use_span_rules: bool = True,
):
    return shared_load_wikidata_definition_candidates(
        query_text,
        use_detailed_description=use_detailed_description,
        exact_match_text=exact_match_text,
        limit=int(candidate_limit),
        filter_name=filter_name,
        require_detailed_description=require_detailed_description,
        target_candidate_count=int(candidate_limit),
        use_span_rules=use_span_rules,
        use_llm_filter=False,
    )


def fetch_raw_candidate_senses_for_merge(word: str, max_candidates: int, use_detailed_description: bool = False):
    """Raw Wikidata candidates (use_span_rules=False), same as the merge experiment."""
    candidates_df, definition_column = load_wikidata_definition_candidates(
        word, use_detailed_description=use_detailed_description, use_span_rules=False, candidate_limit=max_candidates
    )
    candidate_bank = shared_build_wikidata_candidate_bank(candidates_df, definition_column)
    rows = []
    for idx, candidate in enumerate(candidate_bank, start=1):
        rows.append(
            {
                "candidate_id": idx,
                "wikidata_id": candidate["entity_id"],
                "label": candidate["label"],
                "description": candidate["description"],
                "definition": candidate["definition"],
                "hypothesis": candidate["hypothesis"],
            }
        )
    return rows, candidates_df


def compact_candidates_for_merge_prompt(candidates: list) -> list:
    return [
        {"candidate_id": c["candidate_id"], "label": c["label"], "hypothesis": c["hypothesis"]}
        for c in candidates
    ]


def compact_fft_samples_for_merge_prompt(sample_records: list) -> list:
    rows = []
    for sample in sample_records:
        context_text = re.sub(r"\s+", " ", sample["context_text"]).strip()
        sample_order = sample.get("sample_order", sample.get("fft_sample_order"))
        rows.append(
            {
                "sample_order": sample_order,
                "title": sample.get("title"),
                "matched_text": sample["matched_text"],
                "context_text": context_text,
            }
        )
    return rows


def build_merge_prompt_for_candidates(word: str, candidates: list, fft_samples: list) -> str:
    payload = {
        "word": word,
        "candidate_senses": compact_candidates_for_merge_prompt(candidates),
        "fft_dataset_samples": compact_fft_samples_for_merge_prompt(fft_samples),
    }
    return MERGE_WITH_SAMPLES_USER_PROMPT_TEMPLATE.format(
        candidate_payload=json.dumps(payload, ensure_ascii=False, indent=2)
    )


def candidate_senses_to_filter_candidates(candidate_senses: list) -> list:
    return [
        CandidateSense(
            index=int(item["candidate_id"]) - 1,
            entity_id=str(item["wikidata_id"]),
            label=str(item["label"]),
            description=str(item.get("description", "") or ""),
            detailed_description="",
            definition=str(item.get("definition", "") or ""),
            hypothesis=str(item.get("hypothesis", "") or ""),
        )
        for item in candidate_senses
    ]


def merge_candidate_senses_with_fft_samples(word, candidate_senses, fft_samples, llm_filter):
    """LLM merge using the same prompt/method as wikidata_llm_candidate_merge_experiment.ipynb."""
    filter_candidates = candidate_senses_to_filter_candidates(candidate_senses)
    user_prompt = build_merge_prompt_for_candidates(word, candidate_senses, fft_samples)
    chat_kwargs = {"response_format": {"type": "json_object"}}
    model_lower = llm_filter.model.lower()
    if not model_lower.startswith(("gpt-5", "o1", "o3", "o4")):
        chat_kwargs["temperature"] = llm_filter.temperature
    raw_response = llm_filter.llm_client.complete(
        user_prompt, system_prompt=MERGE_WITH_SAMPLES_SYSTEM_PROMPT, **chat_kwargs
    )
    usage = llm_filter.llm_client.last_usage
    api_wait_wall_time = llm_filter.llm_client.last_api_wait_wall_time
    parsed = _extract_json_object(raw_response)
    definitions = _build_definitions(parsed, filter_candidates)
    metadata = {
        "discarded_candidates": parsed.get("discarded_candidates", []),
        "notes": parsed.get("notes", ""),
        "sample_judgments": parsed.get("sample_judgments", []),
        "prompt_style": "wikidata_llm_candidate_merge_experiment",
        "provider": llm_filter.provider,
        "model": llm_filter.model,
        "wikidata_candidate_filter": WIKIDATA_CANDIDATE_MERGE_WITH_SAMPLES_VERSION,
        "candidate_count_after_rule_filter": len(candidate_senses),
        "token_usage": {
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"),
            "total_tokens": usage.get("total_tokens"),
        },
        "api_wait_wall_time": api_wait_wall_time,
    }
    return parsed, definitions, metadata, raw_response


def merged_definitions_to_dataframe(word: str, definitions, metadata: dict) -> pd.DataFrame:
    rows = []
    for rank, defn in enumerate(definitions, start=1):
        primary_id = defn.source_entity_ids[0] if defn.source_entity_ids else f"llm:{word}:{rank}"
        primary_label = defn.canonical_label or (defn.source_labels[0] if defn.source_labels else f"{word} sense {rank}")
        text = defn.definition.strip()
        rows.append(
            {
                "id": primary_id,
                "label": primary_label,
                "description": text,
                "detailed_description": text,
                "match_text": "",
                "aliases": (),
                "concepturi": "",
                "source_entity_ids": tuple(defn.source_entity_ids),
                "source_labels": tuple(defn.source_labels),
                "source_candidate_ids": tuple(defn.source_candidate_ids),
                "canonical_label": defn.canonical_label,
                "merge_rationale": defn.merge_rationale,
                "is_merged": defn.is_merged,
                "is_rewritten": defn.is_rewritten,
                "llm_filter_metadata": metadata,
            }
        )
    return pd.DataFrame(rows)


def build_wikidata_candidate_bank(candidates_df: pd.DataFrame, definition_column: str):
    return shared_build_wikidata_candidate_bank(candidates_df, definition_column)


def build_precomputed_judgments_from_merge_samples(records, sample_positions, merge_result, candidate_bank):
    """Convert combined-merge matched sample_judgments into reusable anchor seed labels."""

    def normalize_sense_id(value) -> str:
        return str(value or "").strip().lower()

    sense_id_to_description = {}
    for item in merge_result.get("merged_senses", []) or []:
        if not isinstance(item, dict):
            continue
        sense_id = normalize_sense_id(item.get("sense_id"))
        description = str(item.get("merged_description", "") or item.get("definition", "") or "").strip()
        if sense_id and description:
            sense_id_to_description[sense_id] = description

    candidate_by_description = {
        candidate["description"]: candidate for candidate in candidate_bank if candidate.get("description")
    }

    judgment_by_order = {}
    for item in merge_result.get("sample_judgments", []) or []:
        if not isinstance(item, dict) or item.get("sample_order") is None:
            continue
        try:
            sample_order = int(item["sample_order"])
        except (TypeError, ValueError):
            continue
        judgment_by_order[sample_order] = item

    precomputed = {}
    label_by_index = {}
    rows = []
    for sample_order, pos in enumerate(sample_positions, start=1):
        record = records[pos]
        record_index = record["record_index"]
        judgment = judgment_by_order.get(sample_order)
        judgment_type = str((judgment or {}).get("judgment", "") or "").strip().lower()
        sense_id = normalize_sense_id((judgment or {}).get("sense_id"))
        description = sense_id_to_description.get(sense_id)
        candidate = candidate_by_description.get(description)
        reason = str((judgment or {}).get("reason", "") or "").strip()

        if candidate is not None and judgment_type == "matched_sense":
            status = "matched"
            precomputed[record_index] = {
                **candidate,
                "score": None,
                "llm_reason": reason,
                "llm_cache_hit": False,
            }
            label_by_index[record_index] = candidate["description"]
        elif judgment is None:
            status = "missing_judgment"
        elif judgment_type != "matched_sense":
            status = judgment_type or "unmatched_judgment"
        elif not description:
            status = "unknown_sense_id"
        else:
            status = "missing_candidate"

        rows.append(
            {
                "sample_order": sample_order,
                "record_position": pos,
                "record_index": record_index,
                "title": record.get("title"),
                "matched_text": record.get("matched_text"),
                "judgment": judgment_type,
                "sense_id": sense_id,
                "assigned_description": description,
                "status": status,
                "reason": reason,
            }
        )

    return precomputed, label_by_index, pd.DataFrame(rows)

In [7]:
text_tokenizer = None
text_encoder_model = None


def load_text_encoder(name_or_path: str, device: str):
    loaded_tokenizer = AutoTokenizer.from_pretrained(
        name_or_path, local_files_only=True, fix_mistral_regex=True, use_fast=True
    )
    if not getattr(loaded_tokenizer, "is_fast", False):
        raise RuntimeError(
            "The text encoder tokenizer must be a fast tokenizer because offset_mapping is required."
        )
    loaded_model = AutoModel.from_pretrained(name_or_path, local_files_only=True)
    loaded_model.to(device)
    loaded_model.eval()
    return loaded_tokenizer, loaded_model


def ensure_text_encoder_loaded(name_or_path: str, device: str):
    global text_tokenizer, text_encoder_model
    if text_tokenizer is None or text_encoder_model is None:
        text_tokenizer, text_encoder_model = load_text_encoder(name_or_path, device)
        print(f"Loaded text encoder on {device}: {name_or_path}")
    return text_tokenizer, text_encoder_model


def encode_span_text_batch(text_list, tokenizer, model, device, max_length=512):
    encoded_inputs = tokenizer(
        text_list, padding=True, truncation=True, max_length=max_length,
        return_offsets_mapping=True, return_tensors="pt",
    )
    offsets = encoded_inputs["offset_mapping"]
    model_inputs = {k: v.to(device) for k, v in encoded_inputs.items() if k != "offset_mapping"}
    with torch.no_grad():
        outputs = model(**model_inputs, output_hidden_states=True)
        token_embeddings = outputs.hidden_states[-2].detach().cpu()
    return token_embeddings, offsets

In [8]:
def collect_query_span_embeddings(
    store, query_text, kind=None, batch_size=DEFAULT_BATCH_SIZE, max_length=DEFAULT_MAX_LENGTH,
    prompt_context_mode=DEFAULT_CONTEXT_MODE, mark_target=False, max_records=None,
    text_encoder_name_or_path=None, device=DEVICE, show_progress=True,
):
    normalized_query = normalize_text(query_text.strip())
    query_records = lookup_records(store, query_text, kind=kind, include_text=True, include_cleaned_text=True)
    if not query_records:
        raise ValueError(f"No records found for span={query_text!r}.")
    if max_records is not None:
        query_records = query_records[: int(max_records)]

    embedding_cache = store.setdefault("embedding_cache", {})
    encoder_name = text_encoder_name_or_path or DEFAULT_TEXT_ENCODER_NAME_OR_PATH
    tokenizer, model = ensure_text_encoder_loaded(encoder_name, device)

    total_records = len(query_records)
    progress_handle = (
        display(f"Embedded 0/{total_records} texts for query {normalized_query!r}", display_id=True)
        if show_progress else None
    )

    embedded_records = []
    cache_hits = 0
    cache_misses = 0

    for batch_start in range(0, total_records, batch_size):
        batch_end = min(batch_start + batch_size, total_records)
        batch_records = query_records[batch_start:batch_end]

        context_texts = []
        local_spans = []
        matched_texts = []
        batch_embeddings = [None] * len(batch_records)
        uncached_texts = []
        uncached_indices = []
        cache_keys = []

        for local_idx, record in enumerate(batch_records):
            prompt_info = build_hotpot_prompt(
                record, query_text, mark_target=mark_target, prompt_context_mode=prompt_context_mode
            )
            cache_key = (
                int(record["document_idx"]),
                tuple(record["span"]),
                prompt_context_mode,
                prompt_info["context_text"],
                tuple(prompt_info["local_span"]),
                int(max_length),
                encoder_name,
                "litsemrag_hidden_states_minus_2_mean_pool_context_only",
            )
            matched_texts.append(prompt_info["matched_text"])
            context_texts.append(prompt_info["context_text"])
            local_spans.append(prompt_info["local_span"])
            cache_keys.append(cache_key)

            cached_embedding = embedding_cache.get(cache_key)
            if cached_embedding is None:
                uncached_texts.append(prompt_info["context_text"])
                uncached_indices.append(local_idx)
                cache_misses += 1
            else:
                batch_embeddings[local_idx] = cached_embedding
                cache_hits += 1

        if uncached_texts:
            token_embeddings_batch, offsets_batch = encode_span_text_batch(
                uncached_texts, tokenizer, model, device, max_length=max_length
            )
            for local_idx, token_embeddings, offsets in zip(uncached_indices, token_embeddings_batch, offsets_batch):
                start_char, end_char = local_spans[local_idx]
                token_indices = get_token_indices_for_phrase(start_char, end_char, offsets.tolist())
                if not token_indices:
                    raise ValueError(
                        f"No tokenizer offsets were found for local_span={(start_char, end_char)} "
                        f"in title={batch_records[local_idx]['title']!r}."
                    )
                embedding = token_embeddings[token_indices].mean(dim=0).to(torch.float32).numpy()
                embedding_cache[cache_keys[local_idx]] = embedding
                batch_embeddings[local_idx] = embedding

        for record, matched_text, context_text, local_span, embedding in zip(
            batch_records, matched_texts, context_texts, local_spans, batch_embeddings
        ):
            prompt_info = build_hotpot_prompt(
                record, query_text, mark_target=mark_target, prompt_context_mode=prompt_context_mode
            )
            item = dict(record)
            item["matched_text"] = matched_text
            item["context_text"] = context_text
            item["local_span"] = local_span
            item["prompt_text"] = prompt_info["prompt_text"]
            item["embedding"] = np.asarray(embedding, dtype=np.float32)
            item["record_index"] = len(embedded_records)
            embedded_records.append(item)

        if progress_handle is not None:
            progress_handle.update(f"Embedded {batch_end}/{total_records} texts for query {normalized_query!r}")

    return {
        "query_text": query_text,
        "normalized_query": normalized_query,
        "kind": kind,
        "record_count": len(embedded_records),
        "prompt_context_mode": prompt_context_mode,
        "max_records": max_records,
        "text_encoder_name_or_path": encoder_name,
        "embedding_method": "LiteSemRAG span mean-pool from hidden_states[-2]",
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
        "records": embedded_records,
    }

In [9]:
def _l2_normalize_rows(matrix):
    norms = np.clip(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12, None)
    return matrix / norms


def sample_anchor_positions(
    records, *, fraction=ANCHOR_FRACTION, fft_ratio=ANCHOR_FFT_RATIO,
    min_count=ANCHOR_MIN_COUNT, random_state=ANCHOR_RANDOM_STATE,
):
    """选 anchor 的位置下标（0..N-1）：fft_ratio 用 FFT 覆盖边界/稀有点，其余 random。"""
    n_records = len(records)
    n_anchor = max(int(min_count), int(round(fraction * n_records)))
    n_anchor = min(n_anchor, n_records)
    n_fft = min(n_anchor, int(round(fft_ratio * n_anchor)))
    n_rand = n_anchor - n_fft

    embeddings = np.stack([r["embedding"] for r in records]).astype(np.float32)
    fft_positions = []
    if n_fft > 0:
        fft_positions = [
            int(p) for p in farthest_first_traversal(embeddings, n_fft, start="random", random_state=random_state)
        ]

    fft_set = set(fft_positions)
    remaining = [p for p in range(n_records) if p not in fft_set]
    rng = np.random.default_rng(random_state)
    rand_positions = []
    if n_rand > 0 and remaining:
        take = min(n_rand, len(remaining))
        rand_positions = [int(p) for p in rng.choice(remaining, size=take, replace=False)]

    anchor_positions = sorted(fft_set | set(rand_positions))
    return {
        "anchor_positions": anchor_positions,
        "n_anchor": len(anchor_positions),
        "n_fft": len(fft_positions),
        "n_random": len(rand_positions),
        "fraction": fraction,
        "fft_ratio": fft_ratio,
    }

In [10]:
def to_label_dict(assignments, key="assigned_description"):
    return {a["record_index"]: a[key] for a in assignments}


def classify_records_with_llm(
    records, candidate_bank, *, cache_path=DEFAULT_LLM_CACHE_PATH,
    context_word_window=DEFAULT_LLM_CONTEXT_WORD_WINDOW, config=None, client=None,
    max_tokens=DEFAULT_LLM_MAX_TOKENS, prompt_mode=LLM_LABEL_PROMPT_MODE,
    max_batch_size=LLM_LABEL_BATCH_SIZE, show_progress=True,
):
    """LLM 在 candidate_bank 上做义项分配（gold）。batch 模式复用 sqlite 缓存并减少 API 调用次数。"""
    llm_client = client or LocalLLMClient(config)
    total_tokens_before = llm_client.total_tokens
    mode = str(prompt_mode or "batch").strip().lower()
    if mode not in {"single", "batch"}:
        raise ValueError(f"Unsupported prompt_mode={prompt_mode!r}; use 'single' or 'batch'.")

    progress_handle = (
        display(f"LLM-labeled 0/{len(records)} records ({mode})", display_id=True) if show_progress else None
    )

    assignments = []
    cache_hit_count = 0
    prompt_tokens_total = None
    completion_tokens_total = None

    if mode == "batch" and int(max_batch_size or 1) > 1:
        batch_records = []
        for record in records:
            span_text = record.get("matched_text") or record.get("query_span") or ""
            batch_records.append(
                {
                    "span_text": span_text,
                    "context_text": record["context_text"],
                    "matched_text": record.get("matched_text") or span_text,
                    "local_span": record.get("local_span"),
                }
            )

        results = choose_wikidata_candidates_with_llm_batch(
            records=batch_records,
            candidate_bank=candidate_bank,
            cache_path=cache_path,
            context_word_window=context_word_window,
            config=config,
            client=llm_client,
            max_tokens=max_tokens,
            max_batch_size=max_batch_size,
        )
        for record, result in zip(records, results):
            if result["cache_hit"]:
                cache_hit_count += 1
            selected_candidate = result["selected_candidate"]
            assignments.append(
                {
                    "record_index": record["record_index"],
                    "assigned_description": selected_candidate.get("description"),
                    "predicted_label": selected_candidate.get("label"),
                    "predicted_entity_id": selected_candidate.get("entity_id"),
                    "predicted_definition": selected_candidate.get("definition"),
                    "provenance": "llm_semantic_label_batch",
                    "llm_reason": result.get("reason"),
                    "llm_cache_hit": bool(result.get("cache_hit")),
                }
            )
        if progress_handle is not None:
            progress_handle.update(
                f"LLM-labeled {len(records)}/{len(records)} records ({mode}, batch_size={max_batch_size})"
            )
    else:
        prompt_tokens_total = 0
        completion_tokens_total = 0
        for idx, record in enumerate(records, start=1):
            result = choose_wikidata_candidate_with_llm(
                span_text=record.get("matched_text") or record.get("query_span") or "",
                context_text=record["context_text"],
                matched_text=record.get("matched_text"),
                local_span=record.get("local_span"),
                candidate_bank=candidate_bank,
                cache_path=cache_path,
                context_word_window=context_word_window,
                config=config,
                client=llm_client,
                max_tokens=max_tokens,
            )
            if result["cache_hit"]:
                cache_hit_count += 1
            else:
                usage = llm_client.last_usage
                if isinstance(usage.get("prompt_tokens"), int):
                    prompt_tokens_total += usage["prompt_tokens"]
                if isinstance(usage.get("completion_tokens"), int):
                    completion_tokens_total += usage["completion_tokens"]
            selected_candidate = result["selected_candidate"]
            assignments.append(
                {
                    "record_index": record["record_index"],
                    "assigned_description": selected_candidate.get("description"),
                    "predicted_label": selected_candidate.get("label"),
                    "predicted_entity_id": selected_candidate.get("entity_id"),
                    "predicted_definition": selected_candidate.get("definition"),
                    "provenance": "llm_semantic_label",
                    "llm_reason": result.get("reason"),
                    "llm_cache_hit": bool(result.get("cache_hit")),
                }
            )
            if progress_handle is not None and (idx % 20 == 0 or idx == len(records)):
                progress_handle.update(f"LLM-labeled {idx}/{len(records)} records ({mode})")

    return {
        "assignments": assignments,
        "cache_hits": cache_hit_count,
        "cache_misses": len(records) - cache_hit_count,
        "provider": llm_client.config.provider,
        "model": llm_client.config.model,
        "prompt_mode": mode,
        "batch_size": int(max_batch_size or 1) if mode == "batch" else 1,
        "prompt_tokens": prompt_tokens_total,
        "completion_tokens": completion_tokens_total,
        "total_tokens": llm_client.total_tokens,
        "total_tokens_delta": llm_client.total_tokens - total_tokens_before,
    }


In [11]:
# 从 LiteSemRAG 索引提取多义词候选（只取词表，不用其义项切分结果），
# 再按 scan store 中的 occurrence 频次排序取 Top-K。
if not INDEX_PKL_PATH.exists():
    raise FileNotFoundError(
        f"Index pickle not found: {INDEX_PKL_PATH}. "
        "请先运行 hotpotqa_latest_framework_index.ipynb 生成索引。"
    )

# 直接 pickle.load（不走 LiteSemRAG.load_data），__setstate__ 只会把运行时句柄置空，
# 不会加载 DeBERTa / spaCy，避免与本 notebook 自己加载的编码器重复占显存。
with INDEX_PKL_PATH.open("rb") as handle:
    index_graph = pickle.load(handle)

multi_sem_words = []
for token_node in index_graph.token_nodes:
    if len(token_node.sem_node_list) >= MULTI_SEM_MIN_SEM_COUNT:
        multi_sem_words.append(token_node.token_text)
multi_sem_words = sorted(set(multi_sem_words))
print(
    f"索引中多义 token（sem_node_list >= {MULTI_SEM_MIN_SEM_COUNT}）: "
    f"{len(multi_sem_words)} / {len(index_graph.token_nodes)}"
)

# 统计每个候选词在 scan store 中的 occurrence 数（按 QUERY_KIND 过滤后）。
def scan_occurrence_count(word, kind=QUERY_KIND):
    normalized = normalize_text(word.strip())
    recs = embedding_store["index"].get(normalized, [])
    if kind is not None:
        recs = [r for r in recs if r["kind"] == kind]
    return len(recs)


word_counts = [(word, scan_occurrence_count(word)) for word in multi_sem_words]
# 过滤掉 scan store 里样本太少 / 不存在的词
word_counts = [(w, c) for (w, c) in word_counts if c >= MIN_OCCURRENCES_FOR_DATASET]
word_counts.sort(key=lambda item: item[1], reverse=True)

selected_words = [w for (w, _) in word_counts[:TOP_K_WORDS]]
selected_word_count_df = pd.DataFrame(word_counts[:TOP_K_WORDS], columns=["word", "scan_occurrences"])

print(
    f"满足 occurrence >= {MIN_OCCURRENCES_FOR_DATASET} 的多义词: {len(word_counts)}；"
    f"按频次取前 {TOP_K_WORDS} 个 -> 实际选中 {len(selected_words)} 个。"
)

# 释放索引对象（仅用于取词表），节省内存。
del index_graph
display(selected_word_count_df)

索引中多义 token（sem_node_list >= 2）: 115 / 59105
满足 occurrence >= 20 的多义词: 115；按频次取前 50 个 -> 实际选中 50 个。


,word,scan_occurrences
0,season,754
1,member,501
2,title,396
3,role,352
4,area,347
5,work,308
6,family,286
7,character,278
8,record,264
9,president,240


In [12]:
# LLM 客户端只建一次，全程复用（同一 sqlite 缓存，re-run 命中即免费）。
gold_llm_config = LocalLLMConfig.from_env(provider=LLM_PROVIDER, model=LLM_MODEL, api_key_file=LLM_API_KEY_FILE)
gold_llm_client = LocalLLMClient(gold_llm_config)

llm_merge_filter = (
    WikidataDefinitionFilter(
        use_api=LLM_WIKIDATA_USE_API,
        api_provider=LLM_WIKIDATA_PROVIDER,
        api_model=LLM_WIKIDATA_MODEL,
        api_key_file=LLM_WIKIDATA_API_KEY_FILE,
        cache_path=str(LLM_WIKIDATA_CACHE_PATH),
    )
    if USE_LLM_WIKIDATA else None
)


def build_word_dataset(word, *, max_records=None, min_records=None):
    """为单个词构建带 gold 标签的数据集条目。失败时抛异常，由外层捕获跳过。"""
    max_records = MAX_OCCURRENCES_PER_WORD if max_records is None else int(max_records)
    min_records = MIN_OCCURRENCES_FOR_DATASET if min_records is None else int(min_records)
    gold_tokens_before = gold_llm_client.total_tokens
    merge_tokens_before = llm_merge_filter.llm_client.total_tokens if llm_merge_filter is not None else 0

    embedded = collect_query_span_embeddings(
        embedding_store, word, kind=QUERY_KIND, max_records=max_records,
        prompt_context_mode=DEFAULT_CONTEXT_MODE, mark_target=MARK_TARGET,
        text_encoder_name_or_path=DEFAULT_TEXT_ENCODER_NAME_OR_PATH, device=DEVICE,
        show_progress=False,
    )
    records = embedded["records"]
    if len(records) < min_records:
        raise ValueError(f"only {len(records)} records after embedding (< {min_records}).")

    # 1) anchor 采样（同时作为候选合并的引导样本）。
    anchor_info = sample_anchor_positions(records)
    anchor_positions = anchor_info["anchor_positions"]
    anchor_samples_for_llm_merge = []
    for sample_order, record_pos in enumerate(anchor_positions, start=1):
        sample = dict(records[record_pos])
        sample["sample_order"] = sample_order
        anchor_samples_for_llm_merge.append(sample)

    # 2) 候选义项库（anchor 引导 LLM 合并，或规则过滤）。
    merge_result = {}
    merge_metadata = {}
    if USE_LLM_WIKIDATA:
        raw_candidate_senses, _ = fetch_raw_candidate_senses_for_merge(
            word, WIKIDATA_CANDIDATE_LIMIT, use_detailed_description=USE_DETAILED_DESCRIPTION
        )
        if not raw_candidate_senses:
            raise ValueError("no raw Wikidata candidates.")
        merge_result, merged_definitions, merge_metadata, _ = merge_candidate_senses_with_fft_samples(
            word, raw_candidate_senses, anchor_samples_for_llm_merge, llm_merge_filter
        )
        if not merged_definitions:
            raise ValueError("LLM merge returned no definitions.")
        candidate_definitions_df = merged_definitions_to_dataframe(word, merged_definitions, merge_metadata)
        definition_column = "description"
    else:
        candidate_definitions_df, definition_column = load_wikidata_definition_candidates(
            word, use_detailed_description=USE_DETAILED_DESCRIPTION, exact_match_text=RULE_EXACT_MATCH_TEXT,
            filter_name=RULE_FILTER_NAME, require_detailed_description=RULE_REQUIRE_DETAILED_DESCRIPTION,
            candidate_limit=WIKIDATA_CANDIDATE_LIMIT, use_span_rules=True,
        )

    merge_tokens_after = llm_merge_filter.llm_client.total_tokens if llm_merge_filter is not None else 0
    candidate_bank = build_wikidata_candidate_bank(candidate_definitions_df, definition_column)
    if not candidate_bank:
        raise ValueError("empty candidate bank.")

    # 3) anchor 样本的 merge 标签（供下游传播算法当 seed 复用）。
    if USE_LLM_WIKIDATA:
        _, anchor_label_by_index, anchor_label_df = build_precomputed_judgments_from_merge_samples(
            records, anchor_positions, merge_result, candidate_bank
        )
    else:
        anchor_label_by_index, anchor_label_df = {}, pd.DataFrame()

    # 4) gold：LLM 逐条标注全部 occurrence。
    gold_result = classify_records_with_llm(
        records, candidate_bank, cache_path=str(LLM_CACHE_PATH), context_word_window=LLM_CONTEXT_WORD_WINDOW,
        config=gold_llm_config, client=gold_llm_client, max_tokens=LLM_MAX_TOKENS,
        prompt_mode=LLM_LABEL_PROMPT_MODE, max_batch_size=LLM_LABEL_BATCH_SIZE, show_progress=False,
    )
    gold_tokens_after = gold_llm_client.total_tokens
    gold_lookup = {a["record_index"]: a for a in gold_result["assignments"]}
    anchor_index_set = {records[pos]["record_index"] for pos in anchor_positions}

    # 5) 组装每条样本。
    sample_rows = []
    for record in records:
        ri = record["record_index"]
        gold = gold_lookup[ri]
        sample_rows.append(
            {
                "record_index": ri,
                "document_idx": int(record["document_idx"]),
                "title": record.get("title"),
                "kind": record.get("kind"),
                "span": tuple(record["span"]),
                "local_span": tuple(record["local_span"]),
                "matched_text": record["matched_text"],
                "context_text": record["context_text"],
                "prompt_text": record["prompt_text"],
                "embedding": np.asarray(record["embedding"], dtype=np.float32),
                "gold_description": gold.get("assigned_description"),
                "gold_label": gold.get("predicted_label"),
                "gold_entity_id": gold.get("predicted_entity_id"),
                "gold_definition": gold.get("predicted_definition"),
                "gold_reason": gold.get("llm_reason"),
                "gold_cache_hit": gold.get("llm_cache_hit"),
                "is_anchor": ri in anchor_index_set,
                "anchor_merge_label": anchor_label_by_index.get(ri),
            }
        )

    gold_labels = [row.get("gold_label") for row in sample_rows if row.get("gold_label") is not None]
    unique_gold_labels = sorted(set(gold_labels))
    if len(unique_gold_labels) <= 1:
        label_hint = unique_gold_labels[0] if unique_gold_labels else None
        raise ValueError(
            f"only one LLM gold_label for {word!r}: {label_hint!r}; skip this word."
        )

    gold_descriptions = [row["gold_description"] for row in sample_rows]
    merge_token_usage = dict((merge_metadata or {}).get("token_usage") or {})
    merge_token_usage.setdefault("total_tokens_delta", merge_tokens_after - merge_tokens_before)
    return {
        "word": word,
        "normalized_word": embedded["normalized_query"],
        "n_records": len(sample_rows),
        "n_gold_senses": len(set(gold_descriptions)),
        "gold_sense_counts": dict(Counter(gold_descriptions)),
        "candidate_bank": candidate_bank,
        "candidate_definitions": candidate_definitions_df.to_dict("records"),
        "anchor_positions": list(anchor_positions),
        "anchor_index_set": sorted(anchor_index_set),
        "anchor_info": anchor_info,
        "anchor_label_by_index": anchor_label_by_index,
        "anchor_label_records": anchor_label_df.to_dict("records") if not anchor_label_df.empty else [],
        "merge_metadata": merge_metadata,
        "merge_sample_judgments": merge_result.get("sample_judgments", []) if USE_LLM_WIKIDATA else [],
        "gold_meta": {
            "provider": gold_result["provider"],
            "model": gold_result["model"],
            "cache_hits": gold_result["cache_hits"],
            "cache_misses": gold_result["cache_misses"],
            "prompt_mode": gold_result.get("prompt_mode"),
            "batch_size": gold_result.get("batch_size"),
            "prompt_tokens": gold_result.get("prompt_tokens"),
            "completion_tokens": gold_result.get("completion_tokens"),
            "total_tokens": gold_result["total_tokens"],
            "total_tokens_delta": gold_result.get("total_tokens_delta", gold_tokens_after - gold_tokens_before),
        },
        "token_usage": {
            "wikidata_merge": merge_token_usage,
            "gold_labeling": {
                "prompt_tokens": gold_result.get("prompt_tokens"),
                "completion_tokens": gold_result.get("completion_tokens"),
                "total_tokens": gold_result.get("total_tokens_delta", gold_tokens_after - gold_tokens_before),
                "total_tokens_delta": gold_result.get("total_tokens_delta", gold_tokens_after - gold_tokens_before),
                "cache_hits": gold_result["cache_hits"],
                "cache_misses": gold_result["cache_misses"],
                "prompt_mode": gold_result.get("prompt_mode"),
                "batch_size": gold_result.get("batch_size"),
            },
            "total_tokens_delta": (merge_tokens_after - merge_tokens_before) + (gold_tokens_after - gold_tokens_before),
        },
        "samples": sample_rows,
    }


def summarize_word_dataset_entry(entry):
    """只展示类别级摘要和 token 消耗，不展开每条样本。"""
    class_rows = []
    for class_id, (description, n_samples) in enumerate(
        Counter(row["gold_description"] for row in entry["samples"]).most_common(), start=1
    ):
        label = next(
            (row.get("gold_label") for row in entry["samples"] if row.get("gold_description") == description),
            None,
        )
        class_rows.append(
            {
                "class_id": class_id,
                "gold_label": label,
                "n_samples": int(n_samples),
                "gold_description": description,
            }
        )
    class_counts_df = pd.DataFrame(class_rows)

    merge_usage = dict(entry.get("token_usage", {}).get("wikidata_merge", {}) or {})
    gold_usage = dict(entry.get("token_usage", {}).get("gold_labeling", {}) or {})
    token_rows = [
        {
            "stage": "wikidata_merge",
            "prompt_tokens": merge_usage.get("prompt_tokens"),
            "completion_tokens": merge_usage.get("completion_tokens"),
            "total_tokens": merge_usage.get("total_tokens"),
            "total_tokens_delta": merge_usage.get("total_tokens_delta", merge_usage.get("total_tokens")),
            "cache_hits": None,
            "cache_misses": None,
        },
        {
            "stage": "gold_labeling",
            "prompt_tokens": gold_usage.get("prompt_tokens"),
            "completion_tokens": gold_usage.get("completion_tokens"),
            "total_tokens": gold_usage.get("total_tokens"),
            "total_tokens_delta": gold_usage.get("total_tokens_delta", gold_usage.get("total_tokens")),
            "cache_hits": gold_usage.get("cache_hits"),
            "cache_misses": gold_usage.get("cache_misses"),
        },
        {
            "stage": "total",
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": entry.get("token_usage", {}).get("total_tokens_delta"),
            "total_tokens_delta": entry.get("token_usage", {}).get("total_tokens_delta"),
            "cache_hits": gold_usage.get("cache_hits"),
            "cache_misses": gold_usage.get("cache_misses"),
        },
    ]
    token_usage_df = pd.DataFrame(token_rows)
    return class_counts_df, token_usage_df


print("build_word_dataset / summarize_word_dataset_entry 已就绪。")


build_word_dataset / summarize_word_dataset_entry 已就绪。


In [13]:
# =============================================================================
# 单词级数据集创建试跑：只展示类别统计与 token 消耗，不展开逐样本明细
# =============================================================================
single_word_test_entry = None
single_word_test_error = None
single_word_class_counts_df = pd.DataFrame()
single_word_token_usage_df = pd.DataFrame()

if RUN_SINGLE_WORD_TEST:
    if TEST_WORD is None:
        if not selected_words:
            raise ValueError("selected_words 为空，无法自动选择测试词。请设置 TEST_WORD。")
        test_word = selected_words[0]
    else:
        test_word = TEST_WORD

    start_time = time.time()
    try:
        single_word_test_entry = build_word_dataset(
            test_word,
            max_records=TEST_MAX_OCCURRENCES_PER_WORD,
            min_records=TEST_MIN_OCCURRENCES_FOR_DATASET,
        )
        single_word_class_counts_df, single_word_token_usage_df = summarize_word_dataset_entry(single_word_test_entry)
        print(
            f"单词试跑完成: {test_word!r} | samples={single_word_test_entry['n_records']} | "
            f"classes={single_word_test_entry['n_gold_senses']} | anchors={single_word_test_entry['anchor_info']['n_anchor']} | "
            f"elapsed={time.time() - start_time:.1f}s"
        )
        print("分类结果摘要：")
        display(single_word_class_counts_df)
        print("token 消耗摘要：")
        display(single_word_token_usage_df)
    except Exception as exc:  # noqa: BLE001 - 试跑时直接展示失败原因，便于换词
        single_word_test_error = {"word": test_word, "error": f"{type(exc).__name__}: {exc}"}
        print(f"单词试跑失败: {single_word_test_error['error']}")
else:
    print("RUN_SINGLE_WORD_TEST=False，跳过单词级试跑。")


RUN_SINGLE_WORD_TEST=False，跳过单词级试跑。


In [14]:
# 逐词构建完整数据集；单词失败（无 Wikidata 候选 / merge 为空等）记录后跳过，不中断整体。
dataset_entries = []
failed_words = []

if RUN_FULL_DATASET_BUILD:
    build_progress = display("Building 0/0 words", display_id=True)
    start_time = time.time()
    for idx, word in enumerate(selected_words, start=1):
        try:
            entry = build_word_dataset(word)
            dataset_entries.append(entry)
            status = f"ok (n={entry['n_records']}, senses={entry['n_gold_senses']})"
        except Exception as exc:  # noqa: BLE001 - 数据集构建需对脏数据稳健
            failed_words.append({"word": word, "error": f"{type(exc).__name__}: {exc}"})
            status = f"SKIP ({type(exc).__name__})"
        build_progress.update(
            f"Building {idx}/{len(selected_words)} words | last={word!r}: {status} | "
            f"ok={len(dataset_entries)} skip={len(failed_words)} | {time.time()-start_time:.0f}s"
        )

    print(f"完成：成功 {len(dataset_entries)} 词，跳过 {len(failed_words)} 词。")
    if failed_words:
        print("跳过明细：")
        display(pd.DataFrame(failed_words))
else:
    print("RUN_FULL_DATASET_BUILD=False，跳过完整数据集构建；如需全量构建请在参数 cell 中改为 True。")

"Building 50/50 words | last='performance': ok (n=90, senses=2) | ok=47 skip=3 | 2011s"

/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Loaded text encoder on cuda: /home/xiaoyue/ProtoGraphRAG/deberta-v3-large
完成：成功 47 词，跳过 3 词。
跳过明细：


,word,error
0,character,ValueError: only one LLM gold_label for 'character': 'fictional character'; skip this word.
1,artist,ValueError: only one LLM gold_label for 'artist': 'artist (creator)'; skip this word.
2,round,JSONDecodeError: Invalid control character at: line 23 column 6461 (char 7617)


In [15]:
# 落盘：完整数据集（含 embedding）用 pickle；概览用 CSV。
def make_non_overwriting_output_paths(pkl_path, csv_path):
    """如果目标文件已存在，自动追加 _rerunNNN，避免覆盖旧数据集。"""
    pkl_path = Path(pkl_path)
    csv_path = Path(csv_path)
    if not pkl_path.exists() and not csv_path.exists():
        return pkl_path, csv_path

    for run_idx in range(1, 1000):
        suffix = f"_rerun{run_idx:03d}"
        candidate_pkl = pkl_path.with_name(f"{pkl_path.stem}{suffix}{pkl_path.suffix}")
        candidate_csv = csv_path.with_name(f"{csv_path.stem}{suffix}{csv_path.suffix}")
        if not candidate_pkl.exists() and not candidate_csv.exists():
            return candidate_pkl, candidate_csv
    raise RuntimeError(f"Cannot find free output filename under {pkl_path.parent}")


SAVED_DATASET_PKL_PATH = None
SAVED_DATASET_SUMMARY_CSV_PATH = None

if dataset_entries:
    output_pkl_path, output_summary_csv_path = make_non_overwriting_output_paths(
        DATASET_PKL_PATH, DATASET_SUMMARY_CSV_PATH
    )
    dataset_blob = {
        "schema_version": 1,
        "task": "hotpotqa_polysemy_sem_propagation_eval",
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "dataset_tag": DATASET_TAG,
        "dataset_run_id": DATASET_RUN_ID,
        "dataset_version_tag": DATASET_VERSION_TAG,
        "output_path": str(output_pkl_path),
        "summary_output_path": str(output_summary_csv_path),
        "params": {
            "index_pkl_path": str(INDEX_PKL_PATH),
            "scan_store_path": str(HOTPOT_SCAN_STORE_PATH),
            "multi_sem_min_sem_count": MULTI_SEM_MIN_SEM_COUNT,
            "top_k_words": TOP_K_WORDS,
            "max_occurrences_per_word": MAX_OCCURRENCES_PER_WORD,
            "min_occurrences_for_dataset": MIN_OCCURRENCES_FOR_DATASET,
            "query_kind": QUERY_KIND,
            "prompt_context_mode": DEFAULT_CONTEXT_MODE,
            "text_encoder": DEFAULT_TEXT_ENCODER_NAME_OR_PATH,
            "embedding_method": "LiteSemRAG span mean-pool from hidden_states[-2]",
            "anchor_fraction": ANCHOR_FRACTION,
            "anchor_fft_ratio": ANCHOR_FFT_RATIO,
            "anchor_min_count": ANCHOR_MIN_COUNT,
            "anchor_random_state": ANCHOR_RANDOM_STATE,
            "wikidata_candidate_limit": WIKIDATA_CANDIDATE_LIMIT,
            "use_llm_wikidata": USE_LLM_WIKIDATA,
            "wikidata_llm_provider": llm_merge_filter.provider if llm_merge_filter is not None else None,
            "wikidata_llm_model": llm_merge_filter.model if llm_merge_filter is not None else None,
            "gold_llm_provider": gold_llm_config.provider,
            "gold_llm_model": gold_llm_config.model,
            "gold_llm_prompt_mode": LLM_LABEL_PROMPT_MODE,
            "gold_llm_batch_size": LLM_LABEL_BATCH_SIZE,
        },
        "words": dataset_entries,
        "failed_words": failed_words,
    }

    with output_pkl_path.open("wb") as handle:
        pickle.dump(dataset_blob, handle, protocol=pickle.HIGHEST_PROTOCOL)
    SAVED_DATASET_PKL_PATH = output_pkl_path
    print(f"已保存数据集: {SAVED_DATASET_PKL_PATH}")

    summary_df = pd.DataFrame(
        [
            {
                "word": e["word"],
                "n_records": e["n_records"],
                "n_gold_senses": e["n_gold_senses"],
                "n_candidates": len(e["candidate_bank"]),
                "n_anchor": e["anchor_info"]["n_anchor"],
                "gold_cache_hits": e["gold_meta"]["cache_hits"],
                "gold_cache_misses": e["gold_meta"]["cache_misses"],
                "total_tokens_delta": e.get("token_usage", {}).get("total_tokens_delta"),
            }
            for e in dataset_entries
        ]
    )
    summary_df.to_csv(output_summary_csv_path, index=False)
    SAVED_DATASET_SUMMARY_CSV_PATH = output_summary_csv_path
    print(f"已保存概览: {SAVED_DATASET_SUMMARY_CSV_PATH}")

    total_samples = int(summary_df["n_records"].sum()) if not summary_df.empty else 0
    print(f"数据集总样本数: {total_samples} | 词数: {len(dataset_entries)}")
    if not summary_df.empty:
        print(f"gold 义项数分布: {dict(Counter(summary_df['n_gold_senses']))}")
    display(summary_df)
else:
    summary_df = pd.DataFrame()
    print("dataset_entries 为空，跳过完整数据集落盘。单词试跑结果保存在 single_word_test_entry 变量中。")


已保存数据集: /home/xiaoyue/LiteSemRAG/data/polysemy_sem_eval/hotpotqa_polysemy_dataset_deepseek_v4_flash_top50_max100_20260605_142830.pkl
已保存概览: /home/xiaoyue/LiteSemRAG/data/polysemy_sem_eval/hotpotqa_polysemy_dataset_deepseek_v4_flash_top50_max100_20260605_142830_summary.csv
数据集总样本数: 4645 | 词数: 47
gold 义项数分布: {2: 17, 3: 21, 4: 6, 5: 2, 7: 1}


,word,n_records,n_gold_senses,n_candidates,n_anchor,gold_cache_hits,gold_cache_misses,total_tokens_delta
0,season,100,2,4,15,0,100,20745
1,member,100,3,3,15,0,100,20261
2,title,100,3,3,15,0,100,28695
3,role,100,3,3,15,0,100,21846
4,area,100,3,3,15,0,100,19705
5,work,100,2,3,15,0,100,20888
6,family,100,3,3,15,0,100,20058
7,record,100,4,4,15,0,100,22102
8,president,100,3,3,15,0,100,20645
9,single,100,2,2,15,0,100,19432


In [16]:
# 重新加载校验：确认 embedding / gold 标签可正常读回。
reload_dataset_path = SAVED_DATASET_PKL_PATH or DATASET_PKL_PATH
if dataset_entries and reload_dataset_path.exists():
    with reload_dataset_path.open("rb") as handle:
        reloaded = pickle.load(handle)

    print(f"schema_version: {reloaded['schema_version']} | words: {len(reloaded['words'])}")
    if reloaded["words"]:
        sample_word = reloaded["words"][0]
        first_sample = sample_word["samples"][0]
        print(f"示例词: {sample_word['word']!r} | n_records={sample_word['n_records']} | "
              f"n_gold_senses={sample_word['n_gold_senses']}")
        print(f"  candidate_bank descriptions: {[c['description'] for c in sample_word['candidate_bank']]}")
        print(f"  样本 embedding shape: {np.asarray(first_sample['embedding']).shape}")
        print(f"  样本 gold_description: {first_sample['gold_description']!r}")
        print(f"  样本 is_anchor: {first_sample['is_anchor']} | matched_text: {first_sample['matched_text']!r}")
else:
    print("没有完整数据集落盘结果，跳过重新加载校验。")


schema_version: 1 | words: 47
示例词: 'season' | n_records=100 | n_gold_senses=2
  candidate_bank descriptions: ['A period of time during which a sports league or competition is held, typically spanning several months and often overlapping two calendar years.', 'A set of episodes of a television series produced and aired as a group.', 'A subdivision of the year based on weather or climate, such as spring, summer, autumn, or winter.', 'A period of the year associated with a particular cultural activity, such as tourism, fashion, or theatre.']
  样本 embedding shape: (1024,)
  样本 gold_description: 'A set of episodes of a television series produced and aired as a group.'
  样本 is_anchor: False | matched_text: 'season'
